In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


DATA_PATH = "synthetic_wsi_delhi_localities.csv" 
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])


# Correlation Heatmap (numeric columns)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr().round(2)
fig = px.imshow(
    corr, text_auto=True, title="Correlation Heatmap (Numeric Features)",
    color_continuous_scale="RdBu_r", aspect="auto"
)
fig.update_layout(height=700)
fig.show()

#Safety Index Distribution
fig = px.histogram(
    df, x="safety_index", nbins=60, marginal="box",
    title="Distribution of Safety Index (Delhi Localities)",
    color_discrete_sequence=["royalblue"]
)
fig.update_layout(bargap=0.05, height=450)
fig.show()

# Safety Index by Time (Hour & Weekend)
df["hour"] = pd.to_datetime(df["timestamp"]).dt.hour
fig = px.box(
    df.sample(n=min(30000, len(df))), x="is_weekend", y="safety_index", color="is_weekend",
    title="Safety Index by Weekend vs Weekday",
    labels={"is_weekend":"Weekend (1=True)"}
)
fig.update_layout(height=500)
fig.show()

fig = px.scatter(
    df.sample(n=min(40000, len(df))), x="hour", y="safety_index",
    color="is_night", title="Safety Index vs Hour of Day (sample)",
    opacity=0.5
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
fig.show()

# Crime Type & Gender Counts
crime_counts = df["crime_type"].value_counts().reset_index()
crime_counts.columns = ["crime_type", "count"]
fig = px.bar(
    crime_counts, x="crime_type", y="count", text="count",
    title="Crime Type Counts (Delhi)",
    color="crime_type"
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=-30, height=500)
fig.show()

vg_counts = df["victim_gender"].value_counts().reset_index()
vg_counts.columns = ["victim_gender", "count"]
fig = px.bar(
    vg_counts, x="victim_gender", y="count", text="count", color="victim_gender",
    title="Victim Gender Distribution"
)
fig.update_traces(textposition='outside')
fig.update_layout(height=400)
fig.show()

# Mean Safety Index by Crime Type
crime_safety = df.groupby("crime_type")["safety_index"].mean().reset_index().sort_values("safety_index", ascending=False)
fig = px.bar(
    crime_safety, x="crime_type", y="safety_index",
    title="Average Safety Index by Crime Type",
    text="safety_index", color="safety_index", color_continuous_scale="RdYlGn"
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=500)
fig.show()

# Feature Relationships
fig = px.scatter(
    df.sample(n=min(30000, len(df))), x="lighting_score", y="safety_index",
    color="crowd_density",
    title="Lighting vs Safety Index (colored by Crowd Density)",
    opacity=0.6
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
fig.show()

fig = px.density_heatmap(
    df.sample(n=min(40000, len(df))),
    x="severity_score", y="safety_index",
    nbinsx=40, nbinsy=40,
    title="Severity vs Safety Index (Density Heatmap)",
    color_continuous_scale="Viridis"
)
fig.update_layout(height=500)
fig.show()

# Spatial Overview: Mean Safety by Locality 
zone_mean = (
    df.groupby("locality", as_index=False)
      .agg({"latitude":"mean","longitude":"mean","safety_index":"mean"})
)
fig = px.scatter_mapbox(
    zone_mean, lat="latitude", lon="longitude",
    color="safety_index", color_continuous_scale="RdYlGn_r",
    size="safety_index", zoom=10.5, height=700,
    hover_data=["locality"],
    title="Average Safety Index by Locality (Delhi)"
)
fig.update_layout(mapbox_style="open-street-map")
fig.show()

# Top/Bottom Localities by Safety 
loc_rank = (
    df.groupby("locality")["safety_index"]
    .mean().reset_index().sort_values("safety_index")
)
fig = px.bar(
    loc_rank.head(25), x="locality", y="safety_index",
    title="25 Localities with Lowest Average Safety Index",
    color="safety_index", color_continuous_scale="Reds"
)
fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

fig = px.bar(
    loc_rank.tail(25), x="locality", y="safety_index",
    title="25 Localities with Highest Average Safety Index",
    color="safety_index", color_continuous_scale="Greens"
)
fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

# Policing vs Safety
fig = px.scatter(
    df.sample(n=min(40000, len(df))), x="policing_index", y="safety_index",
    color="crime_type", opacity=0.5,
    title="Policing Index vs Safety Index (sample)"
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
fig.show()

# Lighting vs Severity (bubble = crowd) 
fig = px.scatter(
    df.sample(n=min(40000, len(df))), x="lighting_score", y="severity_score",
    color="safety_index", size="crowd_density",
    title="Lighting vs Severity (bubble = crowd, color = safety)",
    color_continuous_scale="RdYlGn_r", opacity=0.6
)
fig.update_traces(marker=dict(sizemode="area", sizeref=0.2))
fig.update_layout(height=500)
fig.show()

print("\n EDA complete — visualized safety trends, correlations, and spatial distribution.")
